# Sesión 4 - Práctica: Minimax y Alfa-Beta (EN VIVO)

## Objetivos
- Implementar el juego Tic-Tac-Toe
- Crear una función de evaluación
- Implementar el algoritmo Minimax
- Optimizar con poda Alfa-Beta
- Jugar contra el agente

## Duración estimada
60 minutos

## NOTA IMPORTANTE
Este notebook tiene gaps (huecos) marcados con TODO que completaremos juntos en clase.

## Parte 1: Clase TicTacToe

Primero necesitamos representar el juego y sus reglas.

### GAP 1: Inicializar el tablero (30 seg)

In [1]:
class TicTacToe:
    """
    Clase que representa el juego Tic-Tac-Toe (Tres en Raya).
    
    Tablero representado como lista de 9 elementos:
    [0, 1, 2]
    [3, 4, 5]  ->  [0, 1, 2, 3, 4, 5, 6, 7, 8]
    [6, 7, 8]
    
    Valores:
    0 = vacío
    1 = X (jugador MAX)
    -1 = O (jugador MIN)
    """
    
    def __init__(self):
        """
        Inicializa el tablero vacío.
        """
        self.board = [0] * 9
        self.current_player = 1  # X empieza
    
    def get_moves(self):
        """
        Retorna lista de posiciones vacías (movimientos válidos).
        """
        return [i for i, x in enumerate(self.board) if x == 0]
    
    def make_move(self, pos, player):
        """
        Aplica un movimiento en la posición indicada.
        """
        if self.board[pos] == 0:
            self.board[pos] = player
            return True
        return False
    
    def check_winner(self):
        """
        Verifica si hay un ganador.
        Retorna: 1 (X gana), -1 (O gana), 0 (no hay ganador aún)
        """
        # Filas
        for i in range(0, 9, 3):
            if self.board[i] == self.board[i+1] == self.board[i+2] != 0:
                return self.board[i]
        
        # Columnas
        for i in range(3):
            if self.board[i] == self.board[i+3] == self.board[i+6] != 0:
                return self.board[i]
        
        # Diagonal principal
        if self.board[0] == self.board[4] == self.board[8] != 0:
            return self.board[0]
        
        # Diagonal secundaria
        if self.board[2] == self.board[4] == self.board[6] != 0:
            return self.board[2]
        
        return 0
    
    def is_full(self):
        """
        Verifica si el tablero está lleno.
        """
        return 0 not in self.board
    
    def is_terminal(self):
        """
        Verifica si el juego terminó (hay ganador o empate).
        """
        return self.check_winner() != 0 or self.is_full()
    
    def copy(self):
        """
        Crea una copia del estado actual.
        """
        new_game = TicTacToe()
        new_game.board = self.board.copy()
        new_game.current_player = self.current_player
        return new_game
    
    def display(self):
        """
        Muestra el tablero en formato visual.
        """
        symbols = {0: '.', 1: 'X', -1: 'O'}
        print("\n  0 1 2")
        for row in range(3):
            print(f"{row}", end=" ")
            for col in range(3):
                idx = row * 3 + col
                print(symbols[self.board[idx]], end=" ")
            print()
        print()

### Verificación Gap 1

In [2]:
# Probar inicialización
game = TicTacToe()
print(f"Tablero inicial: {game.board}")
print(f"¿Esperado [0, 0, 0, 0, 0, 0, 0, 0, 0]? {game.board == [0] * 9}")
game.display()

Tablero inicial: [0, 0, 0, 0, 0, 0, 0, 0, 0]
¿Esperado [0, 0, 0, 0, 0, 0, 0, 0, 0]? True

  0 1 2
0 . . . 
1 . . . 
2 . . . 



### GAP 2: Implementar check_winner() (2 min)

Esta es la función más importante del juego.

### Verificación Gap 2

In [3]:
# Test 1: Fila ganadora
game = TicTacToe()
game.board = [1, 1, 1, 0, 0, 0, 0, 0, 0]
game.display()
print(f"Ganador (debe ser 1): {game.check_winner()}")

# Test 2: Columna ganadora
game.board = [1, 0, 0, 1, 0, 0, 1, 0, 0]
game.display()
print(f"Ganador (debe ser 1): {game.check_winner()}")

# Test 3: Diagonal ganadora
game.board = [1, 0, 0, 0, 1, 0, 0, 0, 1]
game.display()
print(f"Ganador (debe ser 1): {game.check_winner()}")

# Test 4: No hay ganador
game.board = [1, -1, 0, 0, 0, 0, 0, 0, 0]
game.display()
print(f"Ganador (debe ser 0): {game.check_winner()}")


  0 1 2
0 X X X 
1 . . . 
2 . . . 

Ganador (debe ser 1): 1

  0 1 2
0 X . . 
1 X . . 
2 X . . 

Ganador (debe ser 1): 1

  0 1 2
0 X . . 
1 . X . 
2 . . X 

Ganador (debe ser 1): 1

  0 1 2
0 X O . 
1 . . . 
2 . . . 

Ganador (debe ser 0): 0


## Parte 2: Función de Evaluación

### GAP 3: Implementar evaluate() (1 min)

In [4]:
def evaluate(game):
    """
    Evalúa el estado actual del juego.
    
    Retorna:
    +10 si X gana (MAX)
    -10 si O gana (MIN)
    0 en cualquier otro caso
    """
    winner = game.check_winner()
    
    if winner == 1:
        return 10  # X gana
    elif winner == -1:
        return -10  # O gana
    else:
        return 0  # Empate o juego en progreso

### Verificación Gap 3

In [5]:
# Test evaluación
game = TicTacToe()

# X gana
game.board = [1, 1, 1, 0, 0, 0, 0, 0, 0]
print(f"X gana - Evaluación (debe ser 10): {evaluate(game)}")

# O gana
game.board = [-1, -1, -1, 0, 0, 0, 0, 0, 0]
print(f"O gana - Evaluación (debe ser -10): {evaluate(game)}")

# Empate
game.board = [1, -1, 1, -1, -1, 1, -1, 1, -1]
print(f"Empate - Evaluación (debe ser 0): {evaluate(game)}")

X gana - Evaluación (debe ser 10): 10
O gana - Evaluación (debe ser -10): -10
Empate - Evaluación (debe ser 0): 0


## Parte 3: Algoritmo Minimax

Ahora implementaremos el algoritmo completo.

### GAP 4: Caso base - Estado terminal (30 seg)

In [ ]:
def minimax(game, depth, is_maximizing):
    """
    Algoritmo Minimax recursivo.
    
    Args:
        game: Estado actual del juego
        depth: Profundidad actual
        is_maximizing: True si turno MAX, False si MIN
    
    Returns:
        Mejor puntuación para el jugador actual
    """
    # Caso base: estado terminal
    # TODO: Si game.is_terminal() es True, retornar evaluate(game)
    if game.is_terminal() or
    
    
    # Caso recursivo: Jugador MAX
    if is_maximizing:
        max_eval = float('-inf')
        
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, 1)  # X = 1
            
            eval_score = minimax(new_game, depth + 1, False)
            max_eval = max(max_eval, eval_score)
        
        return max_eval
    
    # Caso recursivo: Jugador MIN
    else:
        min_eval = float('inf')
        
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, -1)  # O = -1
            
            eval_score = minimax(new_game, depth + 1, True)
            min_eval = min(min_eval, eval_score)
        
        return min_eval

print("Función minimax() básica definida.")

### GAP 5: Maximizar en caso MAX (2 min)

Completar el caso cuando `is_maximizing == True`

### GAP 6: Minimizar en caso MIN (2 min)

Completar el caso cuando `is_maximizing == False`

### Función para encontrar mejor movimiento

In [ ]:
def find_best_move(game):
    """
    Encuentra el mejor movimiento para X (MAX).
    """
    best_val = float('-inf')
    best_move = None
    
    for move in game.get_moves():
        new_game = game.copy()
        new_game.make_move(move, 1)
        
        move_val = minimax(new_game, 0, False)
        
        if move_val > best_val:
            best_val = move_val
            best_move = move
    
    return best_move

print("Función find_best_move() definida.")

### Verificación Minimax

In [ ]:
# Test: X puede ganar en un movimiento
game = TicTacToe()
game.board = [1, 1, 0,    # X X .
              -1, -1, 0,  # O O .
              0, 0, 0]    # . . .

game.display()
best = find_best_move(game)
print(f"Mejor movimiento (debe ser 2 para ganar): {best}")

# Test: O puede ganar, X debe bloquear
game.board = [1, 0, 0,    # X . .
              -1, -1, 0,  # O O .
              0, 0, 0]    # . . .

game.display()
best = find_best_move(game)
print(f"Mejor movimiento (debe ser 5 para bloquear): {best}")

### Contar nodos explorados

In [ ]:
nodes_explored = 0

def minimax_count(game, depth, is_maximizing):
    global nodes_explored
    nodes_explored += 1
    
    if game.is_terminal():
        return evaluate(game)
    
    if is_maximizing:
        max_eval = float('-inf')
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, 1)
            eval_score = minimax_count(new_game, depth + 1, False)
            max_eval = max(max_eval, eval_score)
        return max_eval
    else:
        min_eval = float('inf')
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, -1)
            eval_score = minimax_count(new_game, depth + 1, True)
            min_eval = min(min_eval, eval_score)
        return min_eval

# Contar desde inicio del juego
game = TicTacToe()
nodes_explored = 0
minimax_count(game, 0, True)
print(f"\nNodos explorados con Minimax básico: {nodes_explored:,}")

## Parte 4: Optimización con Alfa-Beta

Ahora implementaremos la poda Alfa-Beta para reducir nodos explorados.

### GAP 7: Actualizar alpha en MAX (30 seg)

In [ ]:
def minimax_alpha_beta(game, depth, alpha, beta, is_maximizing):
    """
    Algoritmo Minimax con poda Alfa-Beta.
    
    Args:
        alpha: Mejor valor para MAX
        beta: Mejor valor para MIN
    """
    if game.is_terminal():
        return evaluate(game)
    
    if is_maximizing:
        max_eval = float('-inf')
        
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, 1)
            
            eval_score = minimax_alpha_beta(new_game, depth + 1, alpha, beta, False)
            max_eval = max(max_eval, eval_score)
            
            # TODO: Actualizar alpha con el máximo entre alpha y max_eval
            
            
            # Poda beta
            if beta <= alpha:
                break
        
        return max_eval
    
    else:
        min_eval = float('inf')
        
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, -1)
            
            eval_score = minimax_alpha_beta(new_game, depth + 1, alpha, beta, True)
            min_eval = min(min_eval, eval_score)
            
            beta = min(beta, min_eval)
            
            # Poda alfa
            if beta <= alpha:
                break
        
        return min_eval

print("Función minimax_alpha_beta() definida.")

### GAP 8: Poda beta en MAX (1 min)

Implementar la condición de poda cuando `beta <= alpha` en el caso MAX.

### GAP 9: Actualizar beta en MIN (30 seg)

### GAP 10: Poda alfa en MIN (1 min)

Implementar la condición de poda cuando `beta <= alpha` en el caso MIN.

### Comparación Minimax vs Alfa-Beta

In [ ]:
nodes_ab = 0

def minimax_alpha_beta_count(game, depth, alpha, beta, is_maximizing):
    global nodes_ab
    nodes_ab += 1
    
    if game.is_terminal():
        return evaluate(game)
    
    if is_maximizing:
        max_eval = float('-inf')
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, 1)
            eval_score = minimax_alpha_beta_count(new_game, depth + 1, alpha, beta, False)
            max_eval = max(max_eval, eval_score)
            alpha = max(alpha, max_eval)
            if beta <= alpha:
                break
        return max_eval
    else:
        min_eval = float('inf')
        for move in game.get_moves():
            new_game = game.copy()
            new_game.make_move(move, -1)
            eval_score = minimax_alpha_beta_count(new_game, depth + 1, alpha, beta, True)
            min_eval = min(min_eval, eval_score)
            beta = min(beta, min_eval)
            if beta <= alpha:
                break
        return min_eval

# Comparar
game = TicTacToe()

nodes_explored = 0
minimax_count(game, 0, True)
minimax_nodes = nodes_explored

nodes_ab = 0
minimax_alpha_beta_count(game, 0, float('-inf'), float('inf'), True)
ab_nodes = nodes_ab

print("\n" + "="*60)
print("COMPARACIÓN DE EFICIENCIA")
print("="*60)
print(f"Minimax básico:       {minimax_nodes:,} nodos")
print(f"Minimax + Alfa-Beta:  {ab_nodes:,} nodos")
print(f"Reducción:            {minimax_nodes - ab_nodes:,} nodos")
print(f"Mejora:               {minimax_nodes / ab_nodes:.1f}x más rápido")
print("="*60)

## Parte 5: Jugar contra el Agente

In [ ]:
def find_best_move_ab(game):
    """
    Encuentra el mejor movimiento usando Alfa-Beta.
    """
    best_val = float('-inf')
    best_move = None
    
    for move in game.get_moves():
        new_game = game.copy()
        new_game.make_move(move, 1)
        
        move_val = minimax_alpha_beta(new_game, 0, float('-inf'), float('inf'), False)
        
        if move_val > best_val:
            best_val = move_val
            best_move = move
    
    return best_move

def play_game():
    """
    Jugar contra el agente (humano es O, agente es X).
    """
    game = TicTacToe()
    print("¡Bienvenido al Tic-Tac-Toe!")
    print("Tú eres O, el agente es X")
    print("Introduce tu movimiento como número de posición (0-8)")
    
    while not game.is_terminal():
        game.display()
        
        # Turno del agente (X)
        print("\nTurno del agente (X)...")
        move = find_best_move_ab(game)
        game.make_move(move, 1)
        print(f"El agente juega en posición {move}")
        
        if game.is_terminal():
            break
        
        # Turno del humano (O)
        game.display()
        while True:
            try:
                move = int(input("\nTu turno (O), posición (0-8): "))
                if move in game.get_moves():
                    game.make_move(move, -1)
                    break
                else:
                    print("Posición inválida. Intenta de nuevo.")
            except ValueError:
                print("Por favor introduce un número entre 0 y 8.")
    
    # Resultado final
    game.display()
    winner = game.check_winner()
    
    if winner == 1:
        print("\n¡El agente (X) gana! 🤖")
    elif winner == -1:
        print("\n¡Ganaste (O)! 🎉")
    else:
        print("\n¡Empate! 🤝")

# Para jugar, ejecuta:
# play_game()

## Resumen

### Has implementado:

✅ **Juego Tic-Tac-Toe completo**  
✅ **Función de evaluación**  
✅ **Algoritmo Minimax** (explora ~549,000 nodos)  
✅ **Poda Alfa-Beta** (reduce a ~18,000 nodos = 30x mejora)  
✅ **Agente invencible**  

### Conceptos clave:

- **Juegos adversariales:** Dos jugadores con objetivos opuestos
- **Minimax:** Estrategia óptima asumiendo oponente perfecto
- **Alfa-Beta:** Poda de ramas que no afectan la decisión
- **Evaluación:** Asignar valor numérico a estados

### Para Actividad 1:

- Implementar Connect-4 o juego similar
- Crear múltiples agentes con estrategias diferentes
- Sistema SPADE para organizar torneo
- Comparar rendimiento de estrategias